# Land

This notebook is an introduction to analyzing CLM land model results from a global simulation.  It uses results from a case similar to the ones you have been running in the Tutorial, but run for longer in order to provide more output. We've prestaged model results from this simulation and another simulation using a different model configuration in a shared directory. This way you can get started on analyzing simulations results and compare differences caused by model structure. 

## Learning Goals

- Read in global CLM data
- Make basic spatial plots
- Make basic timeseries plots

## Running CUPID's LDF (Land Diagnostic Framework) from a CLM checkout

### ONE-TIME instructions on casper or derecho

In [ ]:
git clone -b branch_tags/CUPiD.n01_ctsm5.4.017 https://github.com/ESCOMP/ctsm.git CUPiD.n01_ctsm5.4.017
cd CUPiD.n01_ctsm5.4.017
./bin/git-fleximod update

### LDF ON CASPER

In [ ]:
qsub -I -X -l select=1:ncpus=16:mpiprocs=16:mem=200GB -A <account_num> -q casper@casper-pbs -l walltime=03:00:00
cd tools/CUPiD/examples/land_only
module load conda
conda activate cupid-analysis
module load nco
cp config_subset.yml config.yml	# OR cp config_full.yml config.yml

# For ne30 use ../../externals/LDF/config_clm_unstructured_plots.yaml NEXT
# For f19 use ../../externals/LDF/config_clm_structured_plots.yaml NEXT
../../helper_scripts/generate_ldf_config_file.py --cupid-config-loc . --ldf-template ../../externals/LDF/config_clm_structured_plots.yaml --out-file LDF_config.yaml

# Modify ./LDF_config.yaml with simulation names, years, ...
nohup ../../externals/LDF/run_adf_diag LDF_config.yaml > ldf.out &
cd LDF_output

scp -r <specific_case> <your laptop's desktop>  # this command from your laptop ...and then click on the case, then website, then index.html to view

### ILAMB ON CASPER (same interactive session started by earlier qsub command)

In [ ]:
cd ../external_diag_packages

# Modify ./config.yml with simulation names, years, ...
../../helper_scripts/generate_ilamb_config_files.py --cupid-config-loc . --run-type BGC
# In model_setup.txt, for timeslice simulations set “shift” values 1, 2000 or similar

# Use LDF’S regridded files from earlier
cp -r /glade/derecho/scratch/<user>/cupid_output/<case>/lnd/proc/regrid /glade/derecho/scratch/<user>/archive/<case>/lnd/hist/.

export ILAMB_ROOT=../../ilamb_aux  # ← bash / tcsh → setenv ILAMB_ROOT ../../ilamb_aux

mpiexec ilamb-run --config ./ilamb_nohoff_final_CLM_BGC.cfg --build_dir ./ILAMB_output/ --df_errs ./ilamb_aux/quantiles_Whittaker_cmip5v6.parquet --define_regions ./ilamb_aux/DATA/regions/LandRegions.nc ./ilamb_aux/DATA/regions/Whittaker.nc --regions global --model_setup ./model_setup.txt --filter .clm2.h0 >&! ilamb.out &

scp -r ILAMB_output <your laptop's desktop>  # this command from your laptop ...and then click index.html to view